In [ ]:
!pip install transformers accelerate bitsandbytes pandas tqdm -q

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pandas as pd
import csv
from tqdm import tqdm
import random

print("Loading Llama-3.1-8B-Instruct")

# Configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully")

In [ ]:
def load_data():

    print("Loading dataset")

    all_data = []
    with open('reddit_posts_and_comments_train.csv', 'r', encoding='utf-8', errors='ignore') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            if len(row) == 3:
                all_data.append(row)

    df = pd.DataFrame(all_data, columns=['post_text', 'comment_text', 'stance'])

    neutral_df = df[df['stance'] == 'neutral'].copy()
    concur_df = df[df['stance'] == 'concur'].copy()
    oppose_df = df[df['stance'] == 'oppose'].copy()

    # Randomly sample 100 from each concur/oppose for examples
    concur_sample = concur_df.sample(n=min(100, len(concur_df)), random_state=42)
    oppose_sample = oppose_df.sample(n=min(100, len(oppose_df)), random_state=42)
    oppose_concur_df = pd.concat([concur_sample, oppose_sample])

    print(f"Total samples: {len(df)}")
    print(f"Neutral samples: {len(neutral_df)}")

    return neutral_df, oppose_concur_df

In [ ]:
def build_llama_prompt(post_text, neutral_examples, contrast_examples):

    prompt = f"""You are generating neutral Reddit comments. A neutral comment neither supports nor opposes the post's main point.

CHARACTERISTICS OF NEUTRAL COMMENTS:
- Asks clarifying questions
- Provides related information without taking a stance
- Shares personal experience without agreeing/disagreeing
- Expresses curiosity or seeks more context
- Acknowledges complexity without endorsing a position

NEUTRAL EXAMPLES:
{neutral_examples}

CONTRAST - NON-NEUTRAL EXAMPLES (DO NOT GENERATE LIKE THESE):
{contrast_examples}

NOW GENERATE:
For this post, generate ONE neutral comment that sounds natural and human-like:

Post: {post_text}

Generate only the neutral comment, nothing else. Do not include labels or explanations."""

    return prompt


def generate_neutral_comment(post_text, neutral_df, oppose_concur_df):
    """Generate a single neutral comment"""

    # Sample 5 random neutral examples
    neutral_sample = neutral_df.sample(n=min(5, len(neutral_df)))
    neutral_examples = "\n\n".join([
        f"Post: {row['post_text'][:150]}...\nComment: {row['comment_text']}"
        for _, row in neutral_sample.iterrows()
    ])

    # Sample 3 contrast examples
    contrast_sample = oppose_concur_df.sample(n=min(3, len(oppose_concur_df)))
    contrast_examples = "\n\n".join([
        f"Post: {row['post_text'][:150]}...\nComment ({row['stance']}): {row['comment_text']}"
        for _, row in contrast_sample.iterrows()
    ])

    prompt = build_llama_prompt(post_text, neutral_examples, contrast_examples)

    messages = [
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in generated_text.lower():
        parts = generated_text.split("assistant")
        response = parts[-1].strip()
    else:
        response = generated_text[len(input_text):].strip()

    response = response.strip()

    for prefix in ["Comment:", "Neutral:", "Neutral Comment:", "Response:"]:
        if response.startswith(prefix):
            response = response[len(prefix):].strip()

    return response

In [ ]:
def quality_filter(comment):
    """Basic quality checks"""
    if not comment or len(comment.strip()) < 10:
        return False

    word_count = len(comment.split())
    if word_count < 5 or word_count > 200:
        return False

    strong_words = [
        'completely agree', 'totally agree', 'absolutely right', 'absolutely wrong',
        'completely disagree', 'totally disagree', 'you\'re wrong', 'you\'re right'
    ]

    comment_lower = comment.lower()
    if any(phrase in comment_lower for phrase in strong_words):
        return False

    return True

In [ ]:
def main():

    print("Llama-3.1-8B Synthetic Neutral Comment Generation\n")

    neutral_df, oppose_concur_df = load_data()

    # Configuration
    NUM_SAMPLES = 450
    NUM_RUNS = 10

    all_synthetic_samples = []

    for run in range(NUM_RUNS):
        print(f"RUN {run + 1}/{NUM_RUNS}")

        # Get unique posts
        unique_posts = oppose_concur_df['post_text'].unique()
        random.shuffle(unique_posts)
        posts_to_use = unique_posts[:NUM_SAMPLES]

        print(f"Generating {NUM_SAMPLES} synthetic neutral comments\n")

        # Generate samples
        synthetic_samples = []

        for i, post_text in enumerate(tqdm(posts_to_use, desc=f"Run {run+1}")):
            try:
                # Generate comment
                comment = generate_neutral_comment(post_text, neutral_df, oppose_concur_df)

                # Quality check
                if quality_filter(comment):
                    synthetic_samples.append({
                        'post_text': post_text,
                        'comment_text': comment,
                        'stance': 'neutral'
                    })

                if (i + 1) % 50 == 0:
                    print(f"\nGenerated {len(synthetic_samples)}/{i+1} quality samples in run {run+1}")

            except Exception as e:
                print(f"\nError on sample {i+1}: {e}")
                continue

        print(f"\nRun {run+1}: Generated {len(synthetic_samples)} quality samples")
        all_synthetic_samples.extend(synthetic_samples)

    # Remove duplicates based on comment text
    print(f"\nRemoving duplicates")
    synthetic_df = pd.DataFrame(all_synthetic_samples)
    synthetic_df = synthetic_df.drop_duplicates(subset=['comment_text'], keep='first')

    print("\n" + "=" * 70)
    print(f"Total generated: {len(all_synthetic_samples)} samples")
    print(f"After deduplication: {len(synthetic_df)} unique samples")
    print(f"Filtered out: {NUM_SAMPLES * NUM_RUNS - len(all_synthetic_samples)} low-quality")
    print(f"Duplicates removed: {len(all_synthetic_samples) - len(synthetic_df)}")
    print("=" * 70)

    # Save synthetic data
    synthetic_df.to_csv('synthetic_neutral_llama.csv', index=False)
    print("\nSaved to: synthetic_neutral_llama.csv")

    # Create combined dataset
    combined_df = pd.concat([
        neutral_df,
        oppose_concur_df,
        synthetic_df
    ], ignore_index=True)

    combined_df.to_csv('combined_training_data_llama.csv', index=False)
    print("Saved combined dataset to: combined_training_data_llama.csv")

    # Print final distribution
    print("\nFinal Dataset Distribution:\n")
    print(combined_df['stance'].value_counts())
    print(f"\nTotal samples: {len(combined_df)}")

    # Show sample outputs
    print("\nSample Generated Comments:\n")
    for i, row in synthetic_df.head(3).iterrows():
        print(f"\n{i+1}. Post: {row['post_text'][:100]}...")
        print(f"   Generated: {row['comment_text']}")


In [ ]:
if __name__ == "__main__":
    main()